In [2]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog("default", **{
    "type": "rest",
    "uri": "http://localhost:8181",
    "s3.endpoint": "http://localhost:9000",
    "s3.access-key-id": "minioadmin",
    "s3.secret-access-key": "minioadmin",
    "s3.path-style-access": "true",
    "s3.region": "us-east-1",
})

tbl = catalog.load_table("bronze.who_gho_raw")

df = tbl.scan().to_pandas()

print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

print(df.head())

Total rows: 123110
Columns: ['indicator_code', 'indicator_name', 'country_code', 'country_name', 'year', 'value', 'low', 'high', 'sex', '_ingested_at', '_source_url', '_ingestion_date']
  indicator_code indicator_name country_code country_name  year  value  low  \
0       WHS4_100            NaN          BLR          BLR  2020   97.0  NaN   
1       WHS4_100            NaN          TON          TON  2016   96.0  NaN   
2       WHS4_100            NaN          GRC          GRC  2005   96.0  NaN   
3       WHS4_100            NaN          UKR          UKR  2000   99.0  NaN   
4       WHS4_100            NaN          ERI          ERI  2014   94.0  NaN   

   high  sex                      _ingested_at  \
0   NaN  NaN  2026-05-17T14:30:45.413862+00:00   
1   NaN  NaN  2026-05-17T14:30:45.413862+00:00   
2   NaN  NaN  2026-05-17T14:30:45.413862+00:00   
3   NaN  NaN  2026-05-17T14:30:45.413862+00:00   
4   NaN  NaN  2026-05-17T14:30:45.413862+00:00   

                                 _sour

In [3]:
assert "_ingested_at" in df.columns, "Missing _ingested_at"
assert "_source_url" in df.columns, "Missing _source_url"
assert "_ingestion_date" in df.columns, "Missing _ingestion_date"

print("Ingestion dates found:", df["_ingestion_date"].unique())
print("Indicators found:", df["indicator_code"].unique())
print("Countries found:", df["country_code"].nunique(), "unique")

Ingestion dates found: <ArrowStringArray>
['2026-05-17']
Length: 1, dtype: str
Indicators found: <ArrowStringArray>
['WHS4_100', 'MDG_0000000001', 'WHOSIS_000001']
Length: 3, dtype: str
Countries found: 250 unique


In [4]:
import subprocess
subprocess.run(["python", "ingestion/who_gho.py"])

tbl = catalog.load_table("bronze.who_gho_raw")
snapshots = tbl.history()
print(f"Number of snapshots: {len(snapshots)}")
for snap in snapshots:
    print(snap)

Number of snapshots: 2
snapshot_id=5829151060771756342 timestamp_ms=1779027130004
snapshot_id=275259005088287816 timestamp_ms=1779028287271


In [5]:
first_snapshot_id = snapshots[0].snapshot_id

df_v1 = tbl.scan(snapshot_id=first_snapshot_id).to_pandas()
df_latest = tbl.scan().to_pandas()

print(f"Rows at snapshot 1: {len(df_v1)}")
print(f"Rows at latest:     {len(df_latest)}")
print("Time-travel is working!")

Rows at snapshot 1: 61555
Rows at latest:     123110
Time-travel is working!


In [6]:
import pandas as pd

tb_df = df[df["indicator_code"] == "WHS4_100"].copy()
tb_df = tb_df[tb_df["year"] >= 2015].dropna(subset=["value"])
top10 = (
    tb_df.groupby("country_code")["value"]
    .mean()
    .nlargest(10)
    .reset_index()
)
top10.columns = ["country", "avg_tb_incidence_per_100k"]
print(top10)

  country  avg_tb_incidence_per_100k
0     BRN                       99.0
1     CUB                       99.0
2     HUN                       99.0
3     LUX                       99.0
4     MCO                       99.0
5     NIU                       99.0
6     OMN                       99.0
7     IRN                       98.7
8     PRT                       98.7
9     CHN                       98.5
